In [2]:
import struct
import os
from typing import Dict, List, Tuple
import glob
import shutil
import cv2
import numpy as np
from scipy.spatial.transform import Rotation
from pyquaternion import Quaternion
import open3d as o3d

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [67]:
extrinsics = glob.glob('003_ref/extrinsics/*.txt')
lidar_pose = glob.glob('003_ref/lidar_pose/*.txt')
print(os.path.split(lidar_pose[0])[1])

lidar2world_first = np.loadtxt(lidar_pose[0])

for ext, lidar in zip(extrinsics, lidar_pose):
    cam2world = np.loadtxt(ext)
    lidar2world = np.loadtxt(lidar)
    fname_ext = os.path.split(ext)[1]
    fname_lidar = os.path.split(lidar)[1]

    cam2world_rev = np.linalg.inv(lidar2world_first) @ cam2world
    lidar2world_rev = np.linalg.inv(lidar2world_first) @ lidar2world    
    
    np.savetxt('003_ref/extrinsics_out/'+fname_ext, cam2world_rev) 
    np.savetxt('003_ref/lidar_pose_out/'+fname_lidar, lidar2world_rev)

000.txt


In [61]:
cam2world = np.loadtxt(extrinsics[0])
lidar2world = np.loadtxt(lidar_pose[0])
ext

# Align camera poses with the Lidar pose
cam2world_rev = np.linalg.inv(lidar2world) @ cam2world

print(cam2world_rev)

lidar2world_rev = np.linalg.inv(lidar2world) @ lidar2world

print(lidar2world_rev)

[[ 0.9998968428 -0.0094505834 -0.0108162013 -0.0040958697]
 [ 0.011316328   0.0545757846  0.998445504   0.8425953047]
 [-0.0088455898 -0.9984649068  0.0546771006 -0.306561936 ]
 [ 0.            0.            0.            1.          ]]
[[ 1.0000000000e+00  5.1485640415e-18 -8.0198165577e-18  0.0000000000e+00]
 [-2.1252152915e-18  1.0000000000e+00  8.0220875043e-18 -2.2737367544e-13]
 [-2.1544619364e-18  8.8601659071e-18  1.0000000000e+00  0.0000000000e+00]
 [ 0.0000000000e+00  0.0000000000e+00  0.0000000000e+00  1.0000000000e+00]]


array([[ 1.0000000000e+00,  5.1485640415e-18, -8.0198165577e-18,
         0.0000000000e+00],
       [-2.1252152915e-18,  1.0000000000e+00,  8.0220875043e-18,
        -2.2737367544e-13],
       [-2.1544619364e-18,  8.8601659071e-18,  1.0000000000e+00,
         0.0000000000e+00],
       [ 0.0000000000e+00,  0.0000000000e+00,  0.0000000000e+00,
         1.0000000000e+00]])

In [48]:
extrinsics_cam_to_world = ext

In [56]:
extrinsics_cam_to_world

array([[ 3.3149733422e-02,  6.7805330965e-03,  9.9942739583e-01,
         1.7738672888e+03],
       [-9.9931020751e-01, -1.6523259715e-02,  3.3257947134e-02,
         8.6639038182e+02],
       [ 1.6739305039e-02, -9.9984049041e-01,  6.2281142809e-03,
         1.4906432973e+00],
       [ 0.0000000000e+00,  0.0000000000e+00,  0.0000000000e+00,
         1.0000000000e+00]])

In [57]:
lidar

array([[ 2.2631748905e-02,  9.9858781317e-01,  4.8064366598e-02,
         1.7734128464e+03],
       [-9.9941132647e-01,  2.1358142182e-02,  2.6848282630e-02,
         8.6638416215e+02],
       [ 2.5783802263e-02, -4.8643695968e-02,  9.9848334307e-01,
         1.8370130728e+00],
       [ 0.0000000000e+00,  0.0000000000e+00,  0.0000000000e+00,
         1.0000000000e+00]])

In [36]:
sensor_rotation = np.array([0.5077241387638071,-0.4973392230703816,0.49837167536166627,-0.4964832014373754]) 
sensor_translation = np.array([1.72200568478,0.00475453292289,1.49491291905])
egopose_rotation = np.array([0.9998289065303745,-0.014506321955909566,0.0008505070625267849,0.011445563652516049])
egopose_translation = np.array([1772.1437996767368,866.3028680604585,0.0])

# Extrinsics (camera to ego)
extrinsics_cam_to_ego = np.eye(4)
extrinsics_cam_to_ego[:3, :3] = Quaternion(sensor_rotation).rotation_matrix
extrinsics_cam_to_ego[:3, 3] = np.array(sensor_translation)

# Get ego pose (ego to world)
ego_to_world = np.eye(4)
ego_to_world[:3, :3] = Quaternion(egopose_rotation).rotation_matrix
ego_to_world[:3, 3] = np.array(egopose_translation)

# Transform camera extrinsics to world coordinates
extrinsics_cam_to_world = ego_to_world @ extrinsics_cam_to_ego

print(extrinsics_cam_to_world)


#Lidar_rotation 
sensor_rotation2 = np.array([0.706749235646644,-0.015300993788500868,0.01739745181256607,-0.7070846669051719]) 
sensor_translation2 = np.array([0.985793,0.0,1.84019])
egopose_rotation2 = np.array([0.999828997584073,-0.014381919639738132,0.0010568476587383669,0.011576659731141147])
egopose_translation2 = np.array([1772.4240436762457,866.3084047285619,0.0])

# Extrinsics (Lidar to ego)
extrinsics_cam_to_ego2 = np.eye(4)
extrinsics_cam_to_ego2[:3, :3] = Quaternion(sensor_rotation2).rotation_matrix
extrinsics_cam_to_ego2[:3, 3] = np.array(sensor_translation2)

# Get ego pose (ego to world)
ego_to_world2 = np.eye(4)
ego_to_world2[:3, :3] = Quaternion(egopose_rotation2).rotation_matrix
ego_to_world2[:3, 3] = np.array(egopose_translation2)

# Transform camera extrinsics to world coordinates
extrinsics_lidar_to_world = ego_to_world2 @ extrinsics_cam_to_ego2

print(extrinsics_lidar_to_world)

# Align camera poses with the Lidar pose
cam2world = np.linalg.inv(extrinsics_lidar_to_world) @ extrinsics_cam_to_world

print(cam2world)

lidar2world = np.linalg.inv(extrinsics_lidar_to_world) @ extrinsics_cam_to_world2

print(lidar2world)

[[ 3.3149733422e-02  6.7805330965e-03  9.9942739583e-01  1.7738672888e+03]
 [-9.9931020751e-01 -1.6523259715e-02  3.3257947134e-02  8.6639038182e+02]
 [ 1.6739305039e-02 -9.9984049041e-01  6.2281142809e-03  1.4906432973e+00]
 [ 0.0000000000e+00  0.0000000000e+00  0.0000000000e+00  1.0000000000e+00]]
[[ 2.2631748905e-02  9.9858781317e-01  4.8064366598e-02  1.7734128464e+03]
 [-9.9941132647e-01  2.1358142182e-02  2.6848282630e-02  8.6638416215e+02]
 [ 2.5783802263e-02 -4.8643695968e-02  9.9848334307e-01  1.8370130728e+00]
 [ 0.0000000000e+00  0.0000000000e+00  0.0000000000e+00  1.0000000000e+00]]
[[ 0.9999037794 -0.0091127013 -0.0104589947 -0.0048619201]
 [ 0.0109452486  0.0550539884  0.9984233871  0.4707821621]
 [-0.0085225247 -0.9984417945  0.0551484319 -0.3238349789]
 [ 0.            0.            0.            1.          ]]
[[ 1.0000000000e+00  5.1485640415e-18 -8.0198165577e-18  0.0000000000e+00]
 [-2.1252152915e-18  1.0000000000e+00  8.0220875043e-18 -2.2737367544e-13]
 [-2.154461

[[ 1.0000000000e+00  5.3077158453e-18  4.5103715029e-18  1.1368683772e-13]
 [ 2.2947176869e-19  1.0000000000e+00 -3.9455863103e-19  1.3322676296e-15]
 [-5.1465394671e-18 -2.1464425751e-19  1.0000000000e+00 -2.2737367544e-13]
 [ 0.0000000000e+00  0.0000000000e+00  0.0000000000e+00  1.0000000000e+00]]


In [94]:
rotation = np.array([-0.0015940703021185547,0.99712831150560799,0.074330685542238983,0.014406197300745461]) #XYZW
translation = [0.36718918396685335,-0.5137767828369697,5.1558036640144138]

# Extrinsics (camera to ego)
extrinsics_cam_to_ego = np.eye(4)

#extrinsics_cam_to_ego[:3, :3] = Quaternion(rotation).rotation_matrix
#extrinsics_cam_to_ego[:3, 3] = np.array(translation)
extrinsics_cam_to_ego[:3, :3] = Rotation.from_quat(rotation).as_matrix()
extrinsics_cam_to_ego[:3, 3] = np.array(translation)

#extrincs_cam_to_world = extrinsics_cam_to_ego
extrincs_cam_to_world = np.linalg.inv(extrinsics_cam_to_ego) # colmapのカメラ座標系を世界座標に変換

print(extrincs_cam_to_world)

[[-9.99579841e-01 -1.03734021e-03 -2.89666311e-02  5.15848207e-01]
 [-5.32063030e-03  9.88944816e-01  1.48188533e-01 -2.53980417e-01]
 [ 2.84926777e-02  1.48280391e-01 -9.88534821e-01  5.16241227e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


In [95]:
Rotation.from_quat(rotation).as_matrix()

array([[-0.99957984, -0.00532063,  0.02849268],
       [-0.00103734,  0.98894482,  0.14828039],
       [-0.02896663,  0.14818853, -0.98853482]])

In [96]:
np.linalg.inv(Rotation.from_quat(rotation).as_matrix())

array([[-0.99957984, -0.00103734, -0.02896663],
       [-0.00532063,  0.98894482,  0.14818853],
       [ 0.02849268,  0.14828039, -0.98853482]])

In [97]:
-np.linalg.inv(Rotation.from_quat(rotation).as_matrix()) @ translation

array([ 0.51584821, -0.25398042,  5.16241227])

[0.36718918396685335, -0.5137767828369697, 5.155803664014414]

In [34]:
-0.99957984*0.36718918396685335 -0.00103734*-0.5137767828369697 -0.02896663*5.155803664014414

-0.5158482016395596

In [93]:
rotation = np.array([0.,0.,0.,1.]) #XYZW
#rotation = np.array([0, 0.3826834, 0, 0.9238795 ]) #XYZW
translation = [1.,1.,1.]
Rotation.from_quat(rotation).as_matrix()
np.linalg.inv(Rotation.from_quat(rotation).as_matrix())
np.linalg.inv(Rotation.from_quat(rotation).as_matrix()) @ translation

array([1., 1., 1.])

In [2]:
import math

def quaternion_to_euler(w, x, y, z):
    t0 = +2.0 * (w * x + y * z)
    t1 = +1.0 - 2.0 * (x * x + y * y)
    roll_x = math.atan2(t0, t1)

    t2 = +2.0 * (w * y - z * x)
    t2 = +1.0 if t2 > +1.0 else t2
    t2 = -1.0 if t2 < -1.0 else t2
    pitch_y = math.asin(t2)

    t3 = +2.0 * (w * z + x * y)
    t4 = +1.0 - 2.0 * (y * y + z * z)
    yaw_z = math.atan2(t3, t4)

    deg = math.degrees
    return deg(roll_x), deg(pitch_y), deg(yaw_z)

In [6]:
dic_cameras = {}

with open("CAM_FRONT/sparse/0/cameras.txt") as f:
    for line in f:
        if line[0] != "#":
            info_list = line.strip().split(" ")
            print(info_list[1])
            
            dic_cameras[line[0]] = [float(info_list[4]),float(info_list[5]),float(info_list[6]),float(info_list[7]),0.,0.,0.,0.] 

PINHOLE
['1', 'PINHOLE', '1600', '900', '1244.2634784022885', '1237.9839176673393', '826.58811478139796', '469.98466262245807']


In [7]:
dic_cameras['1']

[1244.2634784022885,
 1237.9839176673393,
 826.588114781398,
 469.9846626224581,
 0.0,
 0.0,
 0.0,
 0.0]

In [8]:
info_list

['1',
 'PINHOLE',
 '1600',
 '900',
 '1244.2634784022885',
 '1237.9839176673393',
 '826.58811478139796',
 '469.98466262245807']

In [20]:
with open("CAM_FRONT/sparse/0/images.txt") as f:
    colmap_images_id = {}
    dic_images_idx = {} # dict初期化
    
    cnt = 0
    for line in f:
        # IMAGE_ID, QW, QX, QY, QZ, TX, TY, TZ, CAMERA_ID, NAME
        if line[0] != "#" and line[-2]=="g":
            
            info_list = line.split(" ")
            if cnt==0:
                print(info_list)
            rotation = np.array([float(info_list[2]),float(info_list[3]),float(info_list[4]),float(info_list[1])]) #XYZW
            
            translation = [float(info_list[5]),float(info_list[6]),float(info_list[7])]
            
            # Extrinsics (camera to ego)
            extrinsics_cam_to_ego = np.eye(4)
            
            #extrinsics_cam_to_ego[:3, :3] = Quaternion(rotation).rotation_matrix
            #extrinsics_cam_to_ego[:3, 3] = np.array(translation)
            extrinsics_cam_to_ego[:3, :3] = Rotation.from_quat(rotation).as_matrix()
            extrinsics_cam_to_ego[:3, 3] = np.array(translation)
                        
            #extrincs_cam_to_world = extrinsics_cam_to_ego
            extrincs_cam_to_world = np.linalg.inv(extrinsics_cam_to_ego) # colmapのカメラ座標系を世界座標に変換
            
            print(extrincs_cam_to_world)
            """
            # Get ego pose (ego to world)
            ego_pose_data = self.nusc.get('ego_pose', cam_data['ego_pose_token'])
            ego_to_world = np.eye(4)
            ego_to_world[:3, :3] = Quaternion(ego_pose_data['rotation']).rotation_matrix
            ego_to_world[:3, 3] = np.array(ego_pose_data['translation'])

            # Transform camera extrinsics to world coordinates
            extrinsics_cam_to_world = ego_to_world @ extrinsics_cam_to_ego
            
            np.savetxt(
                "CAM_FRONT/sparse/0/extrinsics/"
                f"{str(key_frame_idx).zfill(3)}_{str(cam_idx)}.txt",
                extrinsics_cam_to_world
            )
            """
            cnt+=1

['1', '0.99491011748301605', '0.031346092655298766', '0.086173827624140553', '0.041777410616044841', '1.1512940951047796', '0.049288999961785603', '6.8267934938595047', '1', 'n008-2018-08-01-15-16-36-0400__CAM_FRONT__1533151603612404.jpg\n']
[[ 0.98165744  0.08853196 -0.16885131  0.01817295]
 [-0.07772711  0.99454414  0.06957333 -0.43449607]
 [ 0.17408954 -0.05517285  0.98318299 -6.90969607]
 [ 0.          0.          0.          1.        ]]
[[ 0.98190888  0.08809545 -0.16761306  0.01083775]
 [-0.07746846  0.99460912  0.06892996 -0.43078239]
 [ 0.17278189 -0.05469821  0.98344015 -6.86005954]
 [ 0.          0.          0.          1.        ]]
[[ 9.82411742e-01  8.88601121e-02 -1.64228652e-01 -3.57427948e-03]
 [-7.84966373e-02  9.94553809e-01  6.85638269e-02 -4.23598306e-01]
 [ 1.69426820e-01 -5.44665117e-02  9.84036560e-01 -6.76114565e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]
[[ 0.98300554  0.09010205 -0.1599429  -0.01779944]
 [-0.0797595   0.99435613  0

[[ 0.99673901  0.07648697  0.02571154 -0.02292155]
 [-0.07731683  0.99646031  0.03299935  0.01165757]
 [-0.02309651 -0.03487968  0.9991246  -0.02875662]
 [ 0.          0.          0.          1.        ]]
[[ 0.99679314  0.07588589  0.02539246 -0.02174354]
 [-0.07670426  0.99650841  0.03297646  0.01301072]
 [-0.02280136 -0.03481842  0.99913351 -0.00422679]
 [ 0.          0.          0.          1.        ]]
[[ 0.99681505  0.0756514   0.02523152 -0.02122387]
 [-0.07646452  0.99652642  0.03298906  0.01374364]
 [-0.02264821 -0.03481331  0.99913717  0.00786254]
 [ 0.          0.          0.          1.        ]]
[[ 0.99685437  0.07521826  0.02497167 -0.02068326]
 [-0.07602251  0.99656016  0.03299134  0.01494202]
 [-0.02240422 -0.03478597  0.99914363  0.03248332]
 [ 0.          0.          0.          1.        ]]
[[ 0.99691489  0.07459177  0.02442879 -0.01910217]
 [-0.07538188  0.99660192  0.03319907  0.01586456]
 [-0.0218694  -0.03493813  0.99915017  0.07172614]
 [ 0.          0.          

In [19]:
extrincs_cam_to_world

array([[ 0.99680465, -0.0723887 , -0.03376925, -0.0978059 ],
       [ 0.07151299,  0.99708869, -0.0264583 , -0.08893614],
       [ 0.03558621,  0.02395882,  0.99907937, -6.70519276],
       [ 0.        ,  0.        ,  0.        ,  1.        ]])

In [21]:
extrincs_cam_to_world

array([[ 0.99680465,  0.07151299,  0.03558621,  0.3424659 ],
       [-0.0723887 ,  0.99708869,  0.02395882,  0.24224567],
       [-0.03376925, -0.0264583 ,  0.99907937,  6.69336386],
       [ 0.        ,  0.        ,  0.        ,  1.        ]])